# Naive Q/A Evaluation System with RAGAS

## Overview
Baseline question-answering evaluation using Seoul Traffic markdown documents with RAGAS framework

## Goals
- Build baseline Q/A system without complex RAG
- Use KURE-v1 embeddings for Korean text
- Evaluate with EXAONE-3.5-7.8B (best Korean model)
- Generate 50 test questions via RAGAS (25 single-hop + 25 multi-hop)
- Automated evaluation with RAGAS metrics

## Architecture
```
Documents (1,969 MD files)
    ↓
Text Chunks (KURE-v1 embeddings)
    ↓
Chroma Vector Store
    ↓
RAGAS Question Generation (GPT-4o-mini)
    ↓
Q/A System (Retrieval + EXAONE-3.5-7.8B)
    ↓
RAGAS Evaluation (GPT-4o-mini)
```

## Outline

### 1. Environment Setup
- Import libraries:
  - LangChain core + community
  - RAGAS (testset generation + evaluation)
  - KURE-v1 (SentenceTransformer)
  - OpenAI (for RAGAS)
  - Ollama client (for EXAONE)
- Load environment variables (OPENAI_API_KEY)
- Initialize models:
  - KURE-v1 embedding model
  - OpenAI client (gpt-4o-mini for RAGAS)
  - Ollama client (EXAONE-3.5-7.8B)
- Connection tests

## 0. Environment Variables Loading

환경 변수를 `.env` 파일에서 로드합니다.

In [16]:
# ============================================================================
# .env 파일에서 환경 변수 로드
# ============================================================================

from pathlib import Path
from dotenv import load_dotenv
import os

# 프로젝트 루트에서 .env 로드
project_root = Path("/Users/sdh/Dev/02_production_projects/humetro-ai-assistant")
env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(f"❌ .env 파일을 찾을 수 없습니다: {env_path}")

# .env 파일 로드
load_dotenv(env_path)

print("✅ 환경 변수 로드 완료")
print(f"   - .env 경로: {env_path}")

# 필수 환경 변수 확인
required_vars = ["OPENAI_API_KEY"]
missing_vars = [var for var in required_vars if not os.getenv(var)]

if missing_vars:
    raise ValueError(
        f"❌ 필수 환경 변수가 설정되지 않았습니다: {', '.join(missing_vars)}"
    )

print(f"\n📋 환경 변수 확인:")
for var in required_vars:
    value = os.getenv(var)
    masked_value = f"{'*' * (len(value) - 4)}{value[-4:]}" if len(value) > 4 else "****"
    print(f"   - {var}: {masked_value}")


✅ 환경 변수 로드 완료
   - .env 경로: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/.env

📋 환경 변수 확인:
   - OPENAI_API_KEY: ****************************************************************************************************************************************************************ecsA


## Section 1: Environment Setup - Implementation

# ⚠️ KURE-v1 로컬 임베딩 모델 이슈

## 문제 상황
- **증상**: KURE-v1 모델 로딩에 5분 이상 소요
- **원인**:
  - 외장 SD카드 캐시 사용 (`/Volumes/ExternalSD/huggingface`)
  - SentenceTransformer 초기화 오버헤드
  - 2.27GB 모델 크기

## 시도한 해결 방법
1. ✅ MPS 디바이스 명시적 설정 → 효과 미미
2. ⏳ 캐시를 내장 SSD로 이동 → 진행 중
3. ⏳ Lazy loading 패턴 → 지연 해결책

## 임시 해결책 (현재 노트북)
**OpenAI Embeddings 사용으로 전환**
- 모델: `text-embedding-3-small` (1536 차원)
- 장점:
  - 즉시 사용 가능 (API 호출)
  - 로딩 시간 없음
  - 안정적인 성능
- 단점:
  - API 비용 발생 (~$0.02/1M tokens)
  - 네트워크 의존성

## 향후 계획
- KURE-v1 캐시 이동 완료 후 성능 비교 실험
- 로컬 vs OpenAI 임베딩 품질 비교 분석
- 논문에 두 접근 방식 모두 기록

## 영향 범위
- ✅ Section 1: 모델 초기화 → OpenAI embeddings만 사용
- ✅ Section 4: 벡터 스토어 생성 → OpenAI embeddings
- ✅ Section 5: RAGAS 질문 생성 → OpenAI embeddings (기존 계획대로)
- ✅ Section 7: RAGAS 평가 → OpenAI embeddings (기존 계획대로)

---


# ⚠️ GPT-OSS Harmony Format 이슈

## 문제 상황
- **증상**: GPT-OSS 모델이 빈 응답을 생성하거나 출력이 전혀 없음
- **원인**:
  - GPT-OSS는 **Harmony Response Format**을 사용하도록 훈련됨
  - Ollama가 Harmony 포맷을 완벽히 처리하지 못함
  - Structured output 생성 실패 (JSON, 일반 텍스트 모두)

## 기술적 배경
- **Harmony Format**: OpenAI의 gpt-oss 모델 전용 응답 구조
  - `<think>...</think>` 태그로 추론 과정 표현
  - 특수한 대화 구조 정의
  - Function call 구조화
- **Ollama 호환성**:
  - Ollama가 자동 처리해야 하지만 현재 버전에서 불완전
  - GitHub Issues: [#11691](https://github.com/ollama/ollama/issues/11691), [#11781](https://github.com/ollama/ollama/issues/11781)

## 참고 자료
- [OpenAI Cookbook - Harmony Response Format](https://cookbook.openai.com/articles/openai-harmony)
- [GitHub - openai/gpt-oss](https://github.com/openai/gpt-oss)
- [GitHub - openai/harmony](https://github.com/openai/harmony)
- [Medium - What is GPT OSS Harmony Response Format?](https://cobusgreyling.medium.com/what-is-gpt-oss-harmony-response-format-a29f266d6672)

## 시도한 해결 방법
1. ❌ 기본 API 호출 → 빈 응답
2. ❌ `format: "json"` 파라미터 → 파싱 실패
3. ❌ `stop` 토큰 추가 → 여전히 실패

## 현재 해결책
**GPT-OSS 모델 제외**
- 실험에서 GPT-OSS:20b 제거
- 4개 모델로 실험 진행:
  - gpt-5-mini (OpenAI)
  - EXAONE-3.5-7.8B (Ollama)
  - Qwen3-8B (Ollama)
  - Gemma3-12B (Ollama)

## 향후 계획
- Ollama 버전 업데이트 시 재시도
- llama.cpp 픽스가 Ollama에 반영되면 재평가
- 대체 모델 고려:
  - `llama3.3:70b` (Meta)
  - `mixtral:8x7b` (Mistral)
  - `phi4:14b` (Microsoft)

## 영향 범위
- ✅ Section 1.7: Ollama 모델 목록에서 GPT-OSS 제거
- ✅ Section 1.8: 모델 설정에서 GPT-OSS 제외
- ✅ Section 6: Q/A 생성 → 4개 모델 (200 Q/A pairs)
- ✅ Section 7: RAGAS 평가 → 4개 모델
- ✅ 실험 범위: 250개 → 200개 Q/A pairs

---


### 1.1 기본 라이브러리 임포트

Python 표준 라이브러리와 기본 데이터 처리 라이브러리를 임포트합니다.

In [17]:
# 표준 라이브러리
import os
import json
import time
from pathlib import Path

# 데이터 처리
import pandas as pd
import numpy as np

# 진행 상황 추적 (notebook-optimized)
from tqdm.notebook import tqdm

print("✅ 기본 라이브러리 임포트 완료")

✅ 기본 라이브러리 임포트 완료


### 1.2 LangChain 라이브러리 임포트

LangChain 프레임워크의 핵심 컴포넌트들을 임포트합니다.

In [18]:
# LangChain Core
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain Community (Chroma vector store)
from langchain_community.vectorstores import Chroma

# LangChain OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print("✅ LangChain 라이브러리 임포트 완료")

✅ LangChain 라이브러리 임포트 완료


### 1.3 RAGAS 라이브러리 임포트

RAGAS 프레임워크를 사용한 질문 생성 및 평가를 위한 컴포넌트들을 임포트합니다.

In [19]:
# RAGAS (testset generation + evaluation)
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas import evaluate
from ragas.metrics import (
    # Retrieval 메트릭 제외 (동일 retriever 사용으로 차별화 불가)
    # context_precision,  # ❌ 제외: 모든 모델 동일 retriever
    # context_recall,     # ❌ 제외: 모든 모델 동일 retriever
    # Generation 품질 메트릭 (모델별 차이 측정)
    answer_relevancy,  # ✅ 답변이 질문과 관련있는지
    faithfulness,  # ✅ 답변이 context에 충실한지 (hallucination)
    answer_correctness,  # ✅ 답변이 ground truth와 일치하는지
)
from datasets import Dataset

print("✅ RAGAS 라이브러리 임포트 완료")

✅ RAGAS 라이브러리 임포트 완료


### 1.4 디렉토리 설정

데이터 경로와 결과 저장 디렉토리를 설정합니다.

In [20]:
# 디렉토리 설정
BASE_DIR = Path("/Users/sdh/Dev/02_production_projects/humetro-ai-assistant")
DATA_DIR = BASE_DIR / "data/crawled/seoul_traffic/markdown_deduplicated"
VECTORSTORE_DIR = BASE_DIR / "data/vectorstore/seoul_traffic_kure"
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

# 디렉토리 생성
for dir_path in [VECTORSTORE_DIR, RESULTS_DIR, FIGURES_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("✅ 디렉토리 설정 완료")
print(f"   - 데이터: {DATA_DIR}")
print(f"   - 벡터스토어: {VECTORSTORE_DIR}")
print(f"   - 결과: {RESULTS_DIR}")
print(f"   - 그래프: {FIGURES_DIR}")

✅ 디렉토리 설정 완료
   - 데이터: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/data/crawled/seoul_traffic/markdown_deduplicated
   - 벡터스토어: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/data/vectorstore/seoul_traffic_kure
   - 결과: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/results
   - 그래프: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/results/figures


### 1.5 OpenAI 모델 초기화

OpenAI API 클라이언트를 초기화합니다.

In [21]:
print("\n🔧 OpenAI 모델 초기화 중...")

try:
    # Question generation + Answer generation
    gpt5_mini = ChatOpenAI(model="gpt-5-mini", temperature=0.7)

    # Answer evaluation
    gpt5 = ChatOpenAI(model="gpt-5", temperature=0)

    # Embeddings for Vector Store + RAGAS
    openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    print("✅ OpenAI 클라이언트 초기화 완료")
    print("   - Question Gen: gpt-5-mini (temp=0.7)")
    print("   - Answer Gen: gpt-5-mini (temp=0.7)")
    print("   - Evaluation: gpt-5 (temp=0)")
    print("   - Embeddings: text-embedding-3-small (1536 차원)")
except Exception as e:
    print(f"❌ OpenAI 초기화 실패: {e}")
    raise


🔧 OpenAI 모델 초기화 중...
✅ OpenAI 클라이언트 초기화 완료
   - Question Gen: gpt-5-mini (temp=0.7)
   - Answer Gen: gpt-5-mini (temp=0.7)
   - Evaluation: gpt-5 (temp=0)
   - Embeddings: text-embedding-3-small (1536 차원)


### 1.6 KURE-v1 로컬 임베딩 (비활성화)

KURE-v1 로컬 임베딩 모델은 현재 로딩 이슈로 비활성화되었습니다.

In [22]:
print("\n⚠️ KURE-v1 로컬 임베딩 모델...")
print("   - 현재 비활성화됨")
print("   - 이유: 외장 SD카드 캐시로 인한 5분+ 로딩 시간")
print("   - 대안: OpenAI embeddings 사용 (위)")
print("   - 상태: 이슈 문서화 완료, 향후 성능 비교 예정")


⚠️ KURE-v1 로컬 임베딩 모델...
   - 현재 비활성화됨
   - 이유: 외장 SD카드 캐시로 인한 5분+ 로딩 시간
   - 대안: OpenAI embeddings 사용 (위)
   - 상태: 이슈 문서화 완료, 향후 성능 비교 예정


### 1.7 Ollama 로컬 모델 연결 테스트

Ollama 서버에서 제공하는 로컬 LLM 모델들의 연결 상태를 확인합니다.

In [23]:
print("\n🔧 Ollama 모델 연결 테스트...")

OLLAMA_BASE_URL = "http://100.95.220.92:11434"

ollama_models = [
    {
        "name": "EXAONE-3.5-7.8B",
        "model": "exaone3.5:7.8b",
        "description": "한국어 주력",
    },
    {"name": "Qwen3-8B", "model": "qwen3:8b", "description": "복잡한 추론"},
    # {"name": "GPT-OSS-20B", "model": "gpt-oss:20b", "description": "구조화 작업"},  # ❌ Harmony 포맷 이슈로 제외
    {"name": "Gemma3-12B", "model": "gemma3:12b", "description": "범용 fallback"},
]

import requests

ollama_status = []
# ✅ Fixed: Use enumerate instead of tqdm (avoids infinite loop)
for idx, model_info in enumerate(ollama_models):
    print(f"   테스트 중 ({idx + 1}/{len(ollama_models)}): {model_info['name']}...")
    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": model_info["model"],
                "prompt": "테스트",
                "stream": False,
                "options": {"num_predict": 10},
            },
            timeout=30,
        )

        if response.status_code == 200:
            ollama_status.append(
                {
                    "model": model_info["name"],
                    "status": "✅ Connected",
                    "description": model_info["description"],
                }
            )
        else:
            ollama_status.append(
                {
                    "model": model_info["name"],
                    "status": f"❌ HTTP {response.status_code}",
                    "description": model_info["description"],
                }
            )
    except Exception as e:
        ollama_status.append(
            {
                "model": model_info["name"],
                "status": f"❌ {str(e)[:50]}",
                "description": model_info["description"],
            }
        )

# 연결 상태 출력
print("\n📊 Ollama 연결 상태:")
df_ollama = pd.DataFrame(ollama_status)
print(df_ollama.to_string(index=False))

2025-10-30 17:27:48,657 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 100.95.220.92:11434



🔧 Ollama 모델 연결 테스트...
   테스트 중 (1/3): EXAONE-3.5-7.8B...


2025-10-30 17:27:48,800 - urllib3.connectionpool - DEBUG - http://100.95.220.92:11434 "POST /api/generate HTTP/1.1" 200 553
2025-10-30 17:27:48,803 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 100.95.220.92:11434


   테스트 중 (2/3): Qwen3-8B...


2025-10-30 17:27:48,957 - urllib3.connectionpool - DEBUG - http://100.95.220.92:11434 "POST /api/generate HTTP/1.1" 200 431
2025-10-30 17:27:48,958 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 100.95.220.92:11434


   테스트 중 (3/3): Gemma3-12B...


2025-10-30 17:27:49,238 - urllib3.connectionpool - DEBUG - http://100.95.220.92:11434 "POST /api/generate HTTP/1.1" 200 435



📊 Ollama 연결 상태:
          model      status description
EXAONE-3.5-7.8B ✅ Connected      한국어 주력
       Qwen3-8B ✅ Connected      복잡한 추론
     Gemma3-12B ✅ Connected 범용 fallback


### 1.8 모델 설정 요약

실험에서 사용할 모든 모델의 설정을 정리합니다.

In [24]:
# 모델 설정 딕셔너리
models_config = [
    {
        "name": "gpt-5-mini",
        "type": "openai",
        "model": "gpt-5-mini",
        "client": gpt5_mini,
    },
    {
        "name": "EXAONE-3.5-7.8B",
        "type": "ollama",
        "model": "exaone3.5:7.8b",
        "url": OLLAMA_BASE_URL,
    },
    {"name": "Qwen3-8B", "type": "ollama", "model": "qwen3:8b", "url": OLLAMA_BASE_URL},
    # {"name": "GPT-OSS-20B", "type": "ollama", "model": "gpt-oss:20b", "url": OLLAMA_BASE_URL},  # ❌ Harmony 포맷 이슈
    {
        "name": "Gemma3-12B",
        "type": "ollama",
        "model": "gemma3:12b",
        "url": OLLAMA_BASE_URL,
    },
]

print(f"\n✅ 총 4개 모델 설정 완료")
print(f"   - OpenAI: 1개 (gpt-5-mini)")
print(f"   - Ollama: 3개 (EXAONE, Qwen3, Gemma3)")
print(f"\n📝 임베딩 전략:")
print(f"   - Vector Store: OpenAI text-embedding-3-small")
print(f"   - RAGAS: OpenAI text-embedding-3-small")
print(f"   - KURE-v1: 비활성화 (로딩 이슈, 향후 비교 예정)")
print(f"\n⚠️ 제외된 모델:")
print(f"   - GPT-OSS-20B: Harmony Response Format 호환성 이슈")


✅ 총 4개 모델 설정 완료
   - OpenAI: 1개 (gpt-5-mini)
   - Ollama: 3개 (EXAONE, Qwen3, Gemma3)

📝 임베딩 전략:
   - Vector Store: OpenAI text-embedding-3-small
   - RAGAS: OpenAI text-embedding-3-small
   - KURE-v1: 비활성화 (로딩 이슈, 향후 비교 예정)

⚠️ 제외된 모델:
   - GPT-OSS-20B: Harmony Response Format 호환성 이슈


In [25]:
# ============================================================================
# API 가용성 종합 테스트
# ============================================================================

print("\n🔍 API 가용성 종합 테스트...")
print("=" * 80)

api_test_results = []

# Test 1: OpenAI GPT-5-mini (Question Generation + Answer Generation)
print("\n1️⃣ OpenAI GPT-5-mini 테스트...")
try:
    test_response = gpt5_mini.invoke("안녕하세요. 짧게 답변해주세요.")
    api_test_results.append(
        {
            "API": "OpenAI GPT-5-mini",
            "Purpose": "Question + Answer Gen",
            "Status": "✅ Available",
            "Response Length": len(test_response.content),
            "Note": "정상 작동",
        }
    )
    print(f"   ✅ GPT-5-mini 응답 성공 ({len(test_response.content)} chars)")
except Exception as e:
    api_test_results.append(
        {
            "API": "OpenAI GPT-5-mini",
            "Purpose": "Question + Answer Gen",
            "Status": "❌ Failed",
            "Response Length": 0,
            "Note": str(e)[:50],
        }
    )
    print(f"   ❌ GPT-5-mini 실패: {e}")

# Test 2: OpenAI GPT-5 (Answer Evaluation)
print("\n2️⃣ OpenAI GPT-5 테스트...")
try:
    test_response = gpt5.invoke("테스트 메시지입니다.")
    api_test_results.append(
        {
            "API": "OpenAI GPT-5",
            "Purpose": "Answer Evaluation",
            "Status": "✅ Available",
            "Response Length": len(test_response.content),
            "Note": "정상 작동",
        }
    )
    print(f"   ✅ GPT-5 응답 성공 ({len(test_response.content)} chars)")
except Exception as e:
    api_test_results.append(
        {
            "API": "OpenAI GPT-5",
            "Purpose": "Answer Evaluation",
            "Status": "❌ Failed",
            "Response Length": 0,
            "Note": str(e)[:50],
        }
    )
    print(f"   ❌ GPT-5 실패: {e}")

# Test 3: OpenAI Embeddings (Vector Store + RAGAS)
print("\n3️⃣ OpenAI Embeddings 테스트...")
try:
    test_embedding = openai_embeddings.embed_query("테스트 쿼리")
    api_test_results.append(
        {
            "API": "OpenAI Embeddings",
            "Purpose": "Vector Store + RAGAS",
            "Status": "✅ Available",
            "Response Length": len(test_embedding),
            "Note": f"{len(test_embedding)}차원 임베딩",
        }
    )
    print(f"   ✅ OpenAI Embeddings 성공 ({len(test_embedding)} 차원)")
except Exception as e:
    api_test_results.append(
        {
            "API": "OpenAI Embeddings",
            "Purpose": "Vector Store + RAGAS",
            "Status": "❌ Failed",
            "Response Length": 0,
            "Note": str(e)[:50],
        }
    )
    print(f"   ❌ OpenAI Embeddings 실패: {e}")

# Test 4: Ollama Models (4개)
print("\n4️⃣ Ollama 모델 테스트...")
for idx, model_info in enumerate(ollama_models):
    print(f"   테스트 중 ({idx + 1}/{len(ollama_models)}): {model_info['name']}...")
    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": model_info["model"],
                "prompt": "서울 지하철은?",
                "stream": False,
                "options": {"num_predict": 50, "temperature": 0.7},
            },
            timeout=30,
        )

        if response.status_code == 200:
            result = response.json()
            response_text = result.get("response", "")
            api_test_results.append(
                {
                    "API": f"Ollama {model_info['name']}",
                    "Purpose": "Answer Generation",
                    "Status": "✅ Available",
                    "Response Length": len(response_text),
                    "Note": f"한국어 응답: {len(response_text)} chars",
                }
            )
            print(f"      ✅ {model_info['name']}: {len(response_text)} chars")
        else:
            api_test_results.append(
                {
                    "API": f"Ollama {model_info['name']}",
                    "Purpose": "Answer Generation",
                    "Status": f"❌ HTTP {response.status_code}",
                    "Response Length": 0,
                    "Note": response.text[:50],
                }
            )
            print(f"      ❌ {model_info['name']}: HTTP {response.status_code}")
    except Exception as e:
        api_test_results.append(
            {
                "API": f"Ollama {model_info['name']}",
                "Purpose": "Answer Generation",
                "Status": "❌ Failed",
                "Response Length": 0,
                "Note": str(e)[:50],
            }
        )
        print(f"      ❌ {model_info['name']}: {str(e)[:50]}")

# API 테스트 결과 요약
print("\n" + "=" * 80)
print("📊 API 가용성 테스트 결과 요약")
print("=" * 80)

df_api_results = pd.DataFrame(api_test_results)
print(df_api_results.to_string(index=False))

# 가용성 통계
total_apis = len(api_test_results)
available_apis = len([r for r in api_test_results if "✅" in r["Status"]])
failed_apis = total_apis - available_apis

print(f"\n📈 가용성 통계:")
print(f"   - 총 API: {total_apis}개")
print(f"   - 사용 가능: {available_apis}개 ({available_apis / total_apis * 100:.1f}%)")
print(f"   - 실패: {failed_apis}개 ({failed_apis / total_apis * 100:.1f}%)")

# Critical API check (KURE-v1 제외)
critical_apis = ["OpenAI GPT-5-mini", "OpenAI GPT-5", "OpenAI Embeddings"]
critical_status = [r for r in api_test_results if r["API"] in critical_apis]
critical_failed = [r for r in critical_status if "❌" in r["Status"]]

if critical_failed:
    print(f"\n⚠️ 경고: {len(critical_failed)}개의 필수 API가 사용 불가능합니다:")
    for api in critical_failed:
        print(f"   - {api['API']}: {api['Note']}")
    print("\n❌ 실험 진행 불가: 필수 API를 먼저 해결해주세요.")
else:
    print(f"\n✅ 모든 필수 API가 정상 작동합니다!")
    print(f"   - OpenAI GPT-5-mini ✅")
    print(f"   - OpenAI GPT-5 ✅")
    print(f"   - OpenAI Embeddings ✅")
    print(f"\n🎯 Section 1 완료: 실험 진행 가능 ✅")

print("=" * 80)

# Note about KURE-v1
print("\n📝 참고:")
print("   - KURE-v1 로컬 임베딩: 현재 비활성화 (로딩 이슈)")
print("   - 대안: OpenAI embeddings 사용")
print("   - 향후 계획: 캐시 이동 후 성능 비교 실험")


2025-10-30 17:27:49,261 - httpcore.connection - DEBUG - close.started
2025-10-30 17:27:49,261 - httpcore.connection - DEBUG - close.complete
2025-10-30 17:27:49,262 - httpcore.connection - DEBUG - connect_tcp.started host='api.openai.com' port=443 local_address=None timeout=None socket_options=None
2025-10-30 17:27:49,304 - httpcore.connection - DEBUG - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x3158a93d0>
2025-10-30 17:27:49,304 - httpcore.connection - DEBUG - start_tls.started ssl_context=<ssl.SSLContext object at 0x31190f0d0> server_hostname='api.openai.com' timeout=None
2025-10-30 17:27:49,318 - httpcore.connection - DEBUG - start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x30ec5e1e0>
2025-10-30 17:27:49,318 - httpcore.http11 - DEBUG - send_request_headers.started request=<Request [b'POST']>
2025-10-30 17:27:49,318 - httpcore.http11 - DEBUG - send_request_headers.complete
2025-10-30 17:27:49,319 - httpcore.http11 


🔍 API 가용성 종합 테스트...

1️⃣ OpenAI GPT-5-mini 테스트...


2025-10-30 17:27:57,844 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 30 Oct 2025 08:27:58 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'user-w94e6amfgiwcoukibku33j16'), (b'openai-processing-ms', b'7342'), (b'openai-project', b'proj_lIGkj3ABk9zrTJegbgsjckOL'), (b'openai-version', b'2020-10-01'), (b'x-envoy-upstream-service-time', b'7640'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'4000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'3999987'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-tokens', b'0s'), (b'x-request-id', b'req_e91bf71ddae742fba4d8e1bb7d800376'), (b'x-openai-proxy-wasm', b'v0.1'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=31536000

   ✅ GPT-5-mini 응답 성공 (18 chars)

2️⃣ OpenAI GPT-5 테스트...


2025-10-30 17:28:01,732 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 30 Oct 2025 08:28:02 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'user-w94e6amfgiwcoukibku33j16'), (b'openai-processing-ms', b'3123'), (b'openai-project', b'proj_lIGkj3ABk9zrTJegbgsjckOL'), (b'openai-version', b'2020-10-01'), (b'x-envoy-upstream-service-time', b'3318'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'2000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'1999990'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-tokens', b'0s'), (b'x-request-id', b'req_772ccc2657b04756924e47a6fa8787eb'), (b'x-openai-proxy-wasm', b'v0.1'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=31536000

...

      ✅ EXAONE-3.5-7.8B: 100 chars
   테스트 중 (2/3): Qwen3-8B...


2025-10-30 17:28:03,001 - urllib3.connectionpool - DEBUG - http://100.95.220.92:11434 "POST /api/generate HTTP/1.1" 200 845
2025-10-30 17:28:03,003 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 100.95.220.92:11434


      ✅ Qwen3-8B: 216 chars
   테스트 중 (3/3): Gemma3-12B...


2025-10-30 17:28:03,788 - urllib3.connectionpool - DEBUG - http://100.95.220.92:11434 "POST /api/generate HTTP/1.1" 200 879


      ✅ Gemma3-12B: 96 chars

📊 API 가용성 테스트 결과 요약
                   API               Purpose      Status  Response Length              Note
     OpenAI GPT-5-mini Question + Answer Gen ✅ Available               18             정상 작동
          OpenAI GPT-5     Answer Evaluation ✅ Available               29             정상 작동
     OpenAI Embeddings  Vector Store + RAGAS ✅ Available             1536        1536차원 임베딩
Ollama EXAONE-3.5-7.8B     Answer Generation ✅ Available              100 한국어 응답: 100 chars
       Ollama Qwen3-8B     Answer Generation ✅ Available              216 한국어 응답: 216 chars
     Ollama Gemma3-12B     Answer Generation ✅ Available               96  한국어 응답: 96 chars

📈 가용성 통계:
   - 총 API: 6개
   - 사용 가능: 6개 (100.0%)
   - 실패: 0개 (0.0%)

✅ 모든 필수 API가 정상 작동합니다!
   - OpenAI GPT-5-mini ✅
   - OpenAI GPT-5 ✅
   - OpenAI Embeddings ✅

🎯 Section 1 완료: 실험 진행 가능 ✅

📝 참고:
   - KURE-v1 로컬 임베딩: 현재 비활성화 (로딩 이슈)
   - 대안: OpenAI embeddings 사용
   - 향후 계획: 캐시 이동 후 성능 비교 실험


In [26]:
# ============================================================================
# Section 1 요약
# ============================================================================

print("\n" + "=" * 80)
print("✅ Section 1: Environment Setup 완료")
print("=" * 80)
print("\n📋 초기화 완료된 구성 요소:")
print("   1. ✅ 라이브러리 임포트 (tqdm, LangChain, RAGAS, OpenAI, KURE)")
print("   2. ✅ 환경 변수 로드 (OPENAI_API_KEY)")
print("   3. ✅ 디렉토리 설정 (data, vectorstore, results, figures)")
print("   4. ✅ KURE-v1 임베딩 모델 (1024 차원)")
print("   5. ✅ OpenAI 클라이언트 (gpt-5-mini, gpt-5, embeddings)")
print("   6. ✅ Ollama 모델 연결 테스트 (4개)")
print("\n🎯 다음 단계: Section 2 - Data Loading & Preprocessing")
print("=" * 80)



✅ Section 1: Environment Setup 완료

📋 초기화 완료된 구성 요소:
   1. ✅ 라이브러리 임포트 (tqdm, LangChain, RAGAS, OpenAI, KURE)
   2. ✅ 환경 변수 로드 (OPENAI_API_KEY)
   3. ✅ 디렉토리 설정 (data, vectorstore, results, figures)
   4. ✅ KURE-v1 임베딩 모델 (1024 차원)
   5. ✅ OpenAI 클라이언트 (gpt-5-mini, gpt-5, embeddings)
   6. ✅ Ollama 모델 연결 테스트 (4개)

🎯 다음 단계: Section 2 - Data Loading & Preprocessing


### 2. Data Loading & Preprocessing
- Load all markdown files from `data/crawled/seoul_traffic/markdown_deduplicated/`
- Statistics: 1,969 files expected
- Parse markdown:
  - Extract clean text content
  - Remove HTML tags, URLs, special formatting
  - Preserve Korean text integrity
- Create LangChain Document objects with metadata:
  - source: file path
  - file_name: basename
  - doc_id: unique identifier
- Statistics: total docs, total chars, avg length, language distribution

## Section 2: Data Loading & Preprocessing - Implementation

In [27]:
# ============================================================================
# Section 2: Data Loading & Preprocessing
# ============================================================================

print("\n" + "=" * 80)
print("📂 Section 2: 데이터 로딩 및 전처리")
print("=" * 80)

print(f"\n📁 데이터 디렉토리: {DATA_DIR}")
print(f"   - 크롤링 소스: 서울시 교통정보 (마크다운)")

# 마크다운 파일 목록 가져오기
import glob

md_files = sorted(glob.glob(str(DATA_DIR / "*.md")))

print(f"\n📊 발견된 파일:")
print(f"   - 총 파일 수: {len(md_files)}개")

if len(md_files) == 0:
    raise FileNotFoundError(f"❌ {DATA_DIR}에 마크다운 파일이 없습니다!")

# 문서 로딩 with progress bar
documents = []
total_chars = 0

print(f"\n📖 문서 로딩 중...")
for i, file_path in enumerate(tqdm(md_files, desc="마크다운 파일 로딩")):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        # 기본 전처리 (과도한 공백 제거)
        content = "\n".join(
            [line.strip() for line in content.split("\n") if line.strip()]
        )

        # LangChain Document 생성
        doc = Document(
            page_content=content,
            metadata={
                "source": file_path,
                "file_name": Path(file_path).name,
                "doc_id": i,
                "chars": len(content),
            },
        )
        documents.append(doc)
        total_chars += len(content)

        # 500개마다 진행 상황 출력
        if (i + 1) % 500 == 0:
            avg_chars = total_chars / (i + 1)
            print(
                f"   📈 진행: {i + 1}/{len(md_files)} 문서 ({avg_chars:.0f} chars/doc)"
            )

    except Exception as e:
        print(f"   ⚠️ 파일 읽기 실패: {Path(file_path).name} - {e}")
        continue

# 최종 통계
print(f"\n✅ 데이터 로딩 완료")
print(f"   - 성공: {len(documents)}개")
print(f"   - 실패: {len(md_files) - len(documents)}개")
print(f"   - 총 문자 수: {total_chars:,} chars")
print(f"   - 평균 길이: {total_chars / len(documents):.0f} chars/doc")
print(f"   - 최소 길이: {min(doc.metadata['chars'] for doc in documents):,} chars")
print(f"   - 최대 길이: {max(doc.metadata['chars'] for doc in documents):,} chars")

print("\n" + "=" * 80)
print("✅ Section 2 완료: 데이터 로딩 및 전처리")
print("=" * 80)



📂 Section 2: 데이터 로딩 및 전처리

📁 데이터 디렉토리: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/data/crawled/seoul_traffic/markdown_deduplicated
   - 크롤링 소스: 서울시 교통정보 (마크다운)

📊 발견된 파일:
   - 총 파일 수: 1969개

📖 문서 로딩 중...


마크다운 파일 로딩:   0%|          | 0/1969 [00:00<?, ?it/s]

   📈 진행: 500/1969 문서 (3714 chars/doc)
   📈 진행: 1000/1969 문서 (3386 chars/doc)
   📈 진행: 1500/1969 문서 (3102 chars/doc)

✅ 데이터 로딩 완료
   - 성공: 1969개
   - 실패: 0개
   - 총 문자 수: 5,595,691 chars
   - 평균 길이: 2842 chars/doc
   - 최소 길이: 0 chars
   - 최대 길이: 15,281 chars

✅ Section 2 완료: 데이터 로딩 및 전처리


### 3. Text Chunking
- Use RecursiveCharacterTextSplitter:
  - chunk_size: 512 (optimized for KURE-v1)
  - chunk_overlap: 128 (25% overlap)
  - separators: Korean-optimized (\n\n, \n, . , " ")
- Create Document objects with chunk metadata:
  - source: original file
  - chunk_index: position in document
  - chunk_id: unique identifier
- Statistics:
  - Total chunks created
  - Avg chunks per document
  - Chunk length distribution

## Section 3: Text Chunking - Implementation

In [28]:
# ============================================================================
# Section 3: Text Chunking
# ============================================================================

print("\n" + "=" * 80)
print("✂️ Section 3: 텍스트 청킹")
print("=" * 80)

# 청킹 설정
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

print(f"\n⚙️ 청킹 설정:")
print(f"   - Chunk Size: {CHUNK_SIZE} chars")
print(f"   - Chunk Overlap: {CHUNK_OVERLAP} chars")
print(f"   - Splitter: RecursiveCharacterTextSplitter")

# Text Splitter 초기화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

print(f"\n✂️ 문서 청킹 중...")
start_time = time.time()

chunks = []
for doc in tqdm(documents, desc="문서 청킹"):
    doc_chunks = text_splitter.split_documents([doc])
    chunks.extend(doc_chunks)

chunking_time = time.time() - start_time

# 청킹 통계
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print(f"\n✅ 청킹 완료 ({chunking_time:.1f}초)")
print(f"   - 원본 문서: {len(documents):,}개")
print(f"   - 생성된 청크: {len(chunks):,}개")
print(f"   - 평균 청크 수: {len(chunks) / len(documents):.1f} chunks/doc")
print(f"   - 평균 청크 길이: {np.mean(chunk_lengths):.0f} chars")
print(f"   - 최소 청크 길이: {min(chunk_lengths)} chars")
print(f"   - 최대 청크 길이: {max(chunk_lengths)} chars")
print(f"   - 청킹 속도: {len(documents) / chunking_time:.1f} docs/sec")

print("\n" + "=" * 80)
print("✅ Section 3 완료: 텍스트 청킹")
print("=" * 80)



✂️ Section 3: 텍스트 청킹

⚙️ 청킹 설정:
   - Chunk Size: 512 chars
   - Chunk Overlap: 128 chars
   - Splitter: RecursiveCharacterTextSplitter

✂️ 문서 청킹 중...


문서 청킹:   0%|          | 0/1969 [00:00<?, ?it/s]


✅ 청킹 완료 (0.2초)
   - 원본 문서: 1,969개
   - 생성된 청크: 15,120개
   - 평균 청크 수: 7.7 chunks/doc
   - 평균 청크 길이: 427 chars
   - 최소 청크 길이: 4 chars
   - 최대 청크 길이: 512 chars
   - 청킹 속도: 9550.2 docs/sec

✅ Section 3 완료: 텍스트 청킹


### 4. Vector Store Creation
- Generate KURE-v1 embeddings:
  - Batch size: 32
  - Normalize: True (for cosine similarity)
  - Progress tracking
- Create Chroma vector store:
  - Collection name: "seoul_traffic_naive_qa"
  - Persist directory: `data/vectorstore/seoul_traffic_kure`
  - Distance metric: cosine
- Test retrieval:
  - Sample query: "서울 지하철 운행 시간은?"
  - Verify top-k=5 results
  - Check similarity scores

## Section 4: Vector Store Creation - Implementation

In [29]:
# ============================================================================
# Section 4: Vector Store Creation (캐싱 최우선 전략)
# ============================================================================

print("\n" + "=" * 80)
print("🗄️ Section 4: 벡터 스토어 생성")
print("=" * 80)

print(f"\n📍 벡터 스토어 경로: {VECTORSTORE_DIR}")
print(f"   - 임베딩 모델: OpenAI text-embedding-3-small (1536 차원)")
print(f"   - 벡터 DB: Chroma (로컬 persist)")

# ============================================================================
# 캐싱 전략: ALWAYS USE CACHE IF EXISTS
# ============================================================================

print(f"\n🔍 캐시 확인 중...")
cache_exists = VECTORSTORE_DIR.exists() and len(list(VECTORSTORE_DIR.glob("*"))) > 0

if cache_exists:
    print(f"   ✅ 캐시 발견!")
    print(f"   - 경로: {VECTORSTORE_DIR}")

    # 캐시 크기 확인
    import subprocess

    cache_size_result = subprocess.run(
        ["du", "-sh", str(VECTORSTORE_DIR)], capture_output=True, text=True
    )
    cache_size = cache_size_result.stdout.split()[0]
    print(f"   - 크기: {cache_size}")

    # 캐시된 벡터 스토어 로드
    print(f"\n📦 캐시된 벡터 스토어 로딩 중...")
    start_time = time.time()

    vectorstore = Chroma(
        persist_directory=str(VECTORSTORE_DIR), embedding_function=openai_embeddings
    )

    load_time = time.time() - start_time

    # 캐시 통계
    collection_count = vectorstore._collection.count()

    print(f"\n✅ 캐시 로드 완료 ({load_time:.1f}초)")
    print(f"   - 벡터 수: {collection_count:,}개")
    print(f"   - 로드 속도: {collection_count / load_time:.0f} vectors/sec")
    print(f"   - 임베딩 API 호출: 0회 (비용 절감: 100%)")

    # 예상 절감 비용 계산
    saved_tokens = collection_count * 100  # 평균 100 tokens/chunk
    saved_cost = (saved_tokens / 1_000_000) * 0.02  # $0.02/1M tokens
    print(f"   - 절감된 비용: ~${saved_cost:.2f}")
    print(f"   - 절감된 시간: ~{collection_count * 0.1:.0f}초 (예상)")

else:
    print(f"   ❌ 캐시 없음")
    print(f"   - 새로 생성 필요")
    print(f"   - 예상 임베딩 API 호출: ~{len(chunks):,}회")
    print(f"   - 예상 비용: ~${(len(chunks) * 100 / 1_000_000) * 0.02:.2f}")
    print(f"   - 예상 시간: ~{len(chunks) * 0.1:.0f}초")

    # 새 벡터 스토어 생성
    print(f"\n🏗️ 벡터 스토어 생성 중...")
    print(f"   ⚠️ {len(chunks):,}개 청크 임베딩 진행...")
    print(f"   ⏳ 약 {len(chunks) * 0.1 / 60:.1f}분 소요 예상")

    start_time = time.time()

    # Batch 임베딩 (효율성)
    BATCH_SIZE = 100
    vectorstore = None

    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="임베딩 배치 생성"):
        batch = chunks[i : i + BATCH_SIZE]

        if vectorstore is None:
            # 첫 배치로 vectorstore 초기화
            vectorstore = Chroma.from_documents(
                documents=batch,
                embedding=openai_embeddings,
                persist_directory=str(VECTORSTORE_DIR),
            )
        else:
            # 이후 배치 추가
            vectorstore.add_documents(batch)

        # 중간 저장 (안전성)
        if (i + BATCH_SIZE) % 500 == 0:
            vectorstore.persist()
            print(f"   💾 중간 저장: {i + BATCH_SIZE}/{len(chunks)} 청크")

    # 최종 저장
    vectorstore.persist()

    embedding_time = time.time() - start_time
    collection_count = vectorstore._collection.count()

    print(f"\n✅ 벡터 스토어 생성 완료 ({embedding_time:.1f}초)")
    print(f"   - 벡터 수: {collection_count:,}개")
    print(f"   - 임베딩 속도: {collection_count / embedding_time:.1f} vectors/sec")
    print(f"   - API 호출 수: ~{collection_count:,}회")
    print(f"   - 실제 비용: ~${(collection_count * 100 / 1_000_000) * 0.02:.2f}")
    print(f"   - 캐시 저장: {VECTORSTORE_DIR}")

# ============================================================================
# 벡터 스토어 검증
# ============================================================================

print(f"\n🧪 벡터 스토어 검증...")

# 테스트 쿼리
test_query = "서울 지하철 운영 시간은?"
test_results = vectorstore.similarity_search(test_query, k=3)

print(f"   - 테스트 쿼리: '{test_query}'")
print(f"   - 검색 결과: {len(test_results)}개")
print(f"   - 첫 번째 결과 미리보기:")
print(f"     {test_results[0].page_content[:100]}...")

print(f"\n✅ 벡터 스토어 정상 작동 확인")

print("\n" + "=" * 80)
print("✅ Section 4 완료: 벡터 스토어 생성 (캐싱 전략)")
print("=" * 80)

print(f"\n💡 중요:")
print(f"   - 다음 실행부터는 캐시 사용으로 즉시 로드")
print(f"   - 임베딩 API 호출 없음 = 비용 0, 시간 < 5초")
print(f"   - 캐시 삭제 시 재생성 필요")


2025-10-30 17:28:04,139 - httpcore.http11 - DEBUG - send_request_headers.started request=<Request [b'POST']>
2025-10-30 17:28:04,140 - httpcore.http11 - DEBUG - send_request_headers.complete
2025-10-30 17:28:04,140 - httpcore.http11 - DEBUG - send_request_body.started request=<Request [b'POST']>
2025-10-30 17:28:04,140 - httpcore.http11 - DEBUG - send_request_body.complete
2025-10-30 17:28:04,141 - httpcore.http11 - DEBUG - receive_response_headers.started request=<Request [b'POST']>



🗄️ Section 4: 벡터 스토어 생성

📍 벡터 스토어 경로: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/data/vectorstore/seoul_traffic_kure
   - 임베딩 모델: OpenAI text-embedding-3-small (1536 차원)
   - 벡터 DB: Chroma (로컬 persist)

🔍 캐시 확인 중...
   ✅ 캐시 발견!
   - 경로: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/data/vectorstore/seoul_traffic_kure
   - 크기: 178M

📦 캐시된 벡터 스토어 로딩 중...

✅ 캐시 로드 완료 (0.0초)
   - 벡터 수: 15,120개
   - 로드 속도: 6973595 vectors/sec
   - 임베딩 API 호출: 0회 (비용 절감: 100%)
   - 절감된 비용: ~$0.03
   - 절감된 시간: ~1512초 (예상)

🧪 벡터 스토어 검증...


2025-10-30 17:28:04,630 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 30 Oct 2025 08:28:04 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'access-control-allow-origin', b'*'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-model', b'text-embedding-3-small'), (b'openai-organization', b'user-w94e6amfgiwcoukibku33j16'), (b'openai-processing-ms', b'70'), (b'openai-project', b'proj_lIGkj3ABk9zrTJegbgsjckOL'), (b'openai-version', b'2020-10-01'), (b'strict-transport-security', b'max-age=31536000; includeSubDomains; preload'), (b'via', b'envoy-router-78f9c65978-7nhn2'), (b'x-envoy-upstream-service-time', b'310'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'5000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'4999985'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-rat

   - 테스트 쿼리: '서울 지하철 운영 시간은?'
   - 검색 결과: 3개
   - 첫 번째 결과 미리보기:
     * [**메일전송**](https://news.seoul.go.kr/traffic/archives/507433?listpage=1#%EB%A9%94%EC%9D%BC%EC%A0%84...

✅ 벡터 스토어 정상 작동 확인

✅ Section 4 완료: 벡터 스토어 생성 (캐싱 전략)

💡 중요:
   - 다음 실행부터는 캐시 사용으로 즉시 로드
   - 임베딩 API 호출 없음 = 비용 0, 시간 < 5초
   - 캐시 삭제 시 재생성 필요


### 5. RAGAS Question Generation

#### 5.1 Setup RAGAS Components
```python
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
```

#### 5.2 Initialize Generator
- Generator LLM: ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
- Critic LLM: ChatOpenAI(model="gpt-4o-mini", temperature=0)
- Embeddings: OpenAIEmbeddings()

#### 5.3 Generate Questions
- **Single-hop questions (25)**:
  - Use SingleHopSpecificQuerySynthesizer
  - Direct factual questions
  - Answer from 1 document
- **Multi-hop questions (25)**:
  - Use MultiHopAbstractQuerySynthesizer + MultiHopSpecificQuerySynthesizer
  - Complex reasoning questions
  - Require 2+ documents
- **Total: 50 questions with ground truth**

#### 5.4 Question Quality Check
- Validate Korean language quality
- Check question diversity
- Verify ground truth presence
- Save to JSON for reproducibility

### 6. Naive Q/A System Implementation

#### 6.1 Retrieval Setup
- Retriever configuration:
  - search_type: "similarity"
  - k: 5 (top-5 chunks)
  - score_threshold: None (accept all)

#### 6.2 Prompt Template
```python
template = """
당신은 서울시 교통 정보를 제공하는 AI 어시스턴트입니다.

다음 문서들을 참고하여 질문에 답변하세요:

{context}

질문: {question}

답변: (문서 내용을 바탕으로 정확하고 간결하게 답변)
"""
```

#### 6.3 Generation Setup
- Model: EXAONE-3.5-7.8B via Ollama
- Parameters:
  - temperature: 0.7
  - max_tokens: 512
  - top_p: 0.9

#### 6.4 QA Pipeline
For each of 50 questions:
1. Retrieve top-k=5 contexts
2. Format prompt with contexts + question
3. Generate answer with EXAONE
4. Capture:
   - question
   - retrieved_contexts (list of 5)
   - generated_answer
   - ground_truth (from RAGAS)
   - latency (seconds)
   - context_scores (similarity)

#### 6.5 Save Results
- DataFrame with all Q/A pairs
- CSV export: `results/naive_qa_results.csv`
- JSON export with full context: `results/naive_qa_full.json`

## RAGAS 평가 메트릭 전략

### 제외된 메트릭 (Retrieval 관련)
- ❌ **context_precision**: Retriever가 관련 문서를 상위에 배치했는지
  - **제외 이유**: 모든 모델이 동일한 retriever 사용 → 차별화 불가
- ❌ **context_recall**: Ground truth 정보가 retrieved context에 포함되었는지
  - **제외 이유**: 모든 모델이 동일한 retriever 사용 → 차별화 불가

### 사용 메트릭 (Generation 품질)
1. **answer_relevancy** (답변 관련성)
   - 생성된 답변이 질문과 얼마나 관련있는지 측정
   - 모델이 질문의 핵심을 이해하고 적절히 응답했는지 평가

2. **faithfulness** (충실도)
   - 생성된 답변이 제공된 context에 얼마나 충실한지 측정
   - Hallucination(환각) 탐지: context에 없는 정보를 생성했는지 확인

3. **answer_correctness** (답변 정확성)
   - 생성된 답변이 ground truth와 얼마나 일치하는지 측정
   - 의미적 유사도와 factual 정확도를 종합 평가

### 평가 초점
- **모델 간 Generation 품질 비교**
  - gpt-5-mini vs EXAONE-3.5-7.8B vs Qwen3-8B vs GPT-OSS-20B vs Gemma3-12B
- **한국어 생성 능력 평가**
- **Hallucination 경향 분석**
- **질문 유형별 성능 차이** (single-hop vs multi-hop)

---


### 7. RAGAS Evaluation

#### 7.1 Prepare Evaluation Dataset
```python
from datasets import Dataset

eval_data = Dataset.from_dict({
    'question': [...],
    'answer': [...],  # EXAONE generated
    'contexts': [...],  # Retrieved chunks
    'ground_truth': [...]  # RAGAS generated
})
```

#### 7.2 RAGAS Metrics
```python
from ragas import evaluate
from ragas.metrics import (
    context_precision,    # How relevant are retrieved docs?
    context_recall,       # Did we retrieve all needed docs?
    answer_relevancy,     # Is answer relevant to question?
    faithfulness,         # Is answer grounded in context?
    answer_correctness    # Is answer factually correct?
)
```

#### 7.3 Run Evaluation
- Use ChatOpenAI (gpt-4o-mini) as evaluator LLM
- Evaluate all 50 Q/A pairs
- Track API usage and costs

#### 7.4 Metrics Explanation
- **context_precision** (0-1): Proportion of retrieved contexts that are relevant
- **context_recall** (0-1): Proportion of ground truth info present in contexts
- **answer_relevancy** (0-1): How well answer addresses the question
- **faithfulness** (0-1): Answer consistency with retrieved contexts
- **answer_correctness** (0-1): Semantic + factual similarity to ground truth

### 8. Results Analysis & Visualization

#### 8.1 Overall Metrics Summary
```python
# Average scores
- Context Precision: X.XX
- Context Recall: X.XX
- Answer Relevancy: X.XX
- Faithfulness: X.XX
- Answer Correctness: X.XX
- Avg Latency: X.XX seconds
- Avg Answer Length: XXX chars
```

#### 8.2 Single-hop vs Multi-hop Comparison
- Bar chart: Metrics by question complexity
- Hypothesis: Single-hop should score higher

#### 8.3 Performance Distributions
- Histogram: RAGAS metrics distribution
- Box plot: Latency by question type
- Scatter: Answer length vs correctness

#### 8.4 Detailed Results Table
| Question | Type | Answer | Context Precision | Context Recall | Answer Relevancy | Faithfulness | Answer Correctness | Latency |
|----------|------|--------|-------------------|----------------|------------------|--------------|-------------------|----------|
| ... | single | ... | 0.85 | 0.90 | 0.88 | 0.92 | 0.78 | 2.3s |

#### 8.5 Export Results
- CSV: `results/naive_qa_evaluation_summary.csv`
- JSON: `results/naive_qa_evaluation_full.json`
- Plots: `results/figures/` (PNG format)

### 9. Error Analysis & Insights

#### 9.1 Low-Performing Questions
- Identify bottom 10% by answer_correctness
- Analyze common patterns:
  - Retrieval failures (low context_recall)
  - Generation issues (low faithfulness)
  - Multi-hop complexity

#### 9.2 Retrieval Analysis
- Questions with low context_precision:
  - Noisy/irrelevant chunks retrieved
  - Embedding quality issues
- Questions with low context_recall:
  - Missing critical information
  - Chunking strategy problems

#### 9.3 Generation Analysis
- Questions with low faithfulness:
  - Model hallucinations
  - Over-generation beyond context
- Questions with low answer_relevancy:
  - Prompt template issues
  - Model understanding problems

#### 9.4 Limitations Documented
- KURE-v1 embedding coverage
- Simple retrieval (no re-ranking)
- No query expansion/reformulation
- Single-stage retrieval
- No citation/source tracking

### 10. Summary & Recommendations

#### 10.1 Key Findings
- Baseline performance established with RAGAS
- Single-hop vs multi-hop performance gap quantified
- EXAONE-3.5-7.8B Korean generation quality verified
- Retrieval bottlenecks identified

#### 10.2 Comparison to Thesis Goals
- This is "Naive RAG" baseline
- Ready for comparison with:
  - Structured RAG (metadata filtering)
  - Graph RAG (knowledge graph)

#### 10.3 Improvement Recommendations
1. **Retrieval Enhancement**:
   - Add re-ranking stage
   - Hybrid search (dense + sparse)
   - Query expansion

2. **Generation Enhancement**:
   - Chain-of-thought prompting
   - Self-consistency decoding
   - Citation generation

3. **Context Enhancement**:
   - Metadata filtering (date, category)
   - Contextual compression
   - Multi-query retrieval

#### 10.4 Next Experiments
- Structured RAG with metadata
- Graph RAG with Neo4j
- Multi-model comparison

#### 10.5 Thesis Documentation
- Export results for thesis Chapter 4
- RAGAS metrics table for paper
- Baseline established for advanced RAG comparison

## Section 5: RAGAS Question Generation - Implementation

### 5.1 RAGAS 컴포넌트 설정

RAGAS `TestsetGenerator`와 질문 합성기들을 설정합니다.

In [30]:
# ============================================================================# Section 5: RAGAS Question Generation# ============================================================================print("\n" + "="*80)print("🤔 Section 5: RAGAS 질문 생성")print("="*80)# ============================================================================# RAGAS 상세 로깅 설정# ============================================================================import loggingfrom datetime import datetime# 로그 파일 경로log_dir = RESULTS_DIR / "logs"log_dir.mkdir(exist_ok=True)timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")log_file = log_dir / f"ragas_generation_{timestamp}.log"# 로깅 설정logging.basicConfig(    level=logging.DEBUG,    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',    handlers=[        logging.FileHandler(log_file, encoding='utf-8'),        logging.StreamHandler()  # 콘솔에도 출력    ])# RAGAS 로거 설정ragas_logger = logging.getLogger('ragas')ragas_logger.setLevel(logging.DEBUG)# LangChain 로거 설정 (RAGAS가 내부적으로 사용)langchain_logger = logging.getLogger('langchain')langchain_logger.setLevel(logging.INFO)# OpenAI 로거 설정openai_logger = logging.getLogger('openai')openai_logger.setLevel(logging.INFO)print(f"\n📋 RAGAS 상세 로깅 설정 완료")print(f"   - 로그 파일: {log_file}")print(f"   - 로그 레벨: DEBUG")print(f"   - 포함 정보:")print(f"     • RAGAS 내부 동작 (질문 생성 과정)")print(f"     • LangChain 호출 (문서 처리)")print(f"     • OpenAI API 호출 (토큰 사용량)")print(f"     • 에러 및 경고 메시지")# ============================================================================# RAGAS 컴포넌트 설정 (RAGAS 0.3.1 API - LangchainLLMWrapper 필수!)# ============================================================================print("\n⚙️ RAGAS 컴포넌트 설정...")print("   - Generator LLM: gpt-5-mini (temp=0.7)")print("   - Embeddings: OpenAI text-embedding-3-small")print("   - RAGAS 0.3.1: LangchainLLMWrapper 사용")try:    from ragas.llms import LangchainLLMWrapper    from ragas.embeddings import LangchainEmbeddingsWrapper    import openai    # LLM Wrapper (RAGAS 0.3.1 필수!)    generator_llm_raw = ChatOpenAI(model="gpt-5-mini", temperature=0.7)    generator_llm = LangchainLLMWrapper(generator_llm_raw)    # Embeddings Wrapper (RAGAS 0.3.1 필수!)    openai_client = openai.OpenAI()    embeddings = LangchainEmbeddingsWrapper(        OpenAIEmbeddings(model="text-embedding-3-small", client=openai_client)    )    # TestsetGenerator 생성 (RAGAS 0.3.1 API)    testset_generator = TestsetGenerator(        llm=generator_llm,        embedding_model=embeddings,    )    print("\n✅ RAGAS TestsetGenerator 초기화 완료")    print("   - RAGAS 버전: 0.3.1")    print("   - LLM: LangchainLLMWrapper(gpt-5-mini)")    print("   - Embeddings: LangchainEmbeddingsWrapper(text-embedding-3-small)")    logging.info("RAGAS TestsetGenerator initialized successfully (v0.3.1 with wrappers)")except Exception as e:    print(f"\n❌ RAGAS 초기화 실패: {e}")    logging.error(f"RAGAS initialization failed: {e}", exc_info=True)    raise

2025-10-30 17:28:04,684 - root - INFO - RAGAS TestsetGenerator initialized successfully (v0.3.1)



🤔 Section 5: RAGAS 질문 생성

📋 RAGAS 상세 로깅 설정 완료
   - 로그 파일: /Users/sdh/Dev/02_production_projects/humetro-ai-assistant/results/logs/ragas_generation_20251030_172804.log
   - 로그 레벨: DEBUG
   - 포함 정보:
     • RAGAS 내부 동작 (질문 생성 과정)
     • LangChain 호출 (문서 처리)
     • OpenAI API 호출 (토큰 사용량)
     • 에러 및 경고 메시지

⚙️ RAGAS 컴포넌트 설정...
   - Generator LLM: gpt-5-mini (temp=0.7)
   - Embeddings: OpenAI text-embedding-3-small

✅ RAGAS TestsetGenerator 초기화 완료
   - RAGAS 버전: 0.3.1
   - LLM: gpt-5-mini (단일 LLM)
   - Embeddings: text-embedding-3-small


### 5.2 질문 합성기 설정

Single-hop과 Multi-hop 질문 생성을 위한 합성기들을 설정합니다.

In [34]:
print("\n⚙️ 질문 합성기 설정...")

# ============================================================================
# Best Practice: default_query_distribution 사용
# ============================================================================
from ragas.testset.synthesizers import default_query_distribution

# RAGAS 권장 방식: LLM 기반 기본 분포 사용
query_distribution = default_query_distribution(generator_llm)

print("\n✅ 질문 합성기 설정 완료")
print("   - Query Distribution: default_query_distribution(generator_llm)")
print("   - 자동으로 single-hop/multi-hop 분포 설정")

# query_distribution 타입 확인 및 출력
print(f"\n📊 분포 타입: {type(query_distribution)}")

if isinstance(query_distribution, dict):
    print("📊 분포 상세:")
    for synthesizer, weight in query_distribution.items():
        print(f"   - {synthesizer.__class__.__name__}: {weight:.1%}")
elif isinstance(query_distribution, list):
    print("📊 Synthesizers 목록:")
    for idx, synthesizer in enumerate(query_distribution, 1):
        print(f"   {idx}. {synthesizer.__class__.__name__}")
else:
    print(f"📊 Query Distribution: {query_distribution}")



⚙️ 질문 합성기 설정...

✅ 질문 합성기 설정 완료
   - Query Distribution: default_query_distribution(generator_llm)
   - 자동으로 single-hop/multi-hop 분포 설정

📊 분포 타입: <class 'list'>
📊 Synthesizers 목록:
   1. tuple
   2. tuple
   3. tuple


### 5.2.5 사용자 페르소나 정의 (선택사항)

다양한 사용자 유형을 반영한 질문 생성을 위해 페르소나를 정의할 수 있습니다.

In [35]:
# ============================================================================\n# 한국어 페르소나 정의 (RAGAS 0.3.1+ 방식)\n# ============================================================================\n\nprint("\\n👥 한국어 페르소나 정의...")\nprint("   ⚠️  중요: 모든 페르소나는 '반드시 한국어로' 질문/답변 생성")\n\nfrom ragas.testset.persona import Persona\n\n# 서울 교통 도메인 페르소나 (한국어 명시)\npersona_first_time_user = Persona(\n    name="지하철 처음 이용자",\n    role_description=\"\"\"\n    지하철을 처음 이용하는 승객입니다.\n    모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.\n    기본적인 이용 방법, 요금, 노선 정보 등에 대해 자세한 안내가 필요합니다.\n    질문은 구체적이고 단계별 설명을 요구하는 형태로 작성됩니다.\n    예: \"지하철 요금은 어떻게 계산되나요?\", \"환승은 어떻게 하나요?\"\n    \"\"\"\n)\n\npersona_frequent_traveler = Persona(\n    name="자주 이용하는 승객",\n    role_description=\"\"\"\n    지하철을 자주 이용하는 승객입니다.\n    모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.\n    효율적인 이동, 시간 절약, 편의시설 위치 등 실용적인 정보에 관심이 많습니다.\n    질문은 간결하고 핵심적인 정보를 요구하는 형태로 작성됩니다.\n    예: \"가장 빠른 환승 경로는?\", \"막차 시간은 언제인가요?\"\n    \"\"\"\n)\n\npersona_dissatisfied = Persona(\n    name="불만을 가진 승객",\n    role_description=\"\"\"\n    서비스나 시설에 불만을 가진 승객입니다.\n    모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.\n    문제 상황에 대한 즉각적인 해결책과 보상을 요구합니다.\n    질문은 다소 감정적이고 즉각적인 조치를 요구하는 형태로 작성됩니다.\n    예: \"지연 보상은 어떻게 받나요?\", \"불편사항을 어디에 신고하나요?\"\n    \"\"\"\n)\n\npersona_transportation_vulnerable = Persona(\n    name="교통약자",\n    role_description=\"\"\"\n    어르신, 장애인, 임산부 등 교통약자입니다.\n    모든 질문과 답변은 반드시 한국어로 작성되어야 합니다.\n    접근성, 편의시설, 도움 서비스 등에 대한 정보가 필요합니다.\n    질문은 이해하기 쉽고 배려를 요구하는 형태로 작성됩니다.\n    예: \"휠체어로 이용 가능한가요?\", \"도움이 필요하면 어디로 연락하나요?\"\n    \"\"\"\n)\n\npersonas = [\n    persona_first_time_user,\n    persona_frequent_traveler,\n    persona_dissatisfied,\n    persona_transportation_vulnerable,\n]\n\nprint(f"\\n✅ {len(personas)}개 한국어 페르소나 정의 완료")\nfor persona in personas:\n    print(f"   - {persona.name}")\n\nprint(f"\\n🎯 핵심 특징:")\nprint(f"   - 모든 페르소나에 '반드시 한국어로' 명시")\nprint(f"   - 서울 지하철 도메인 특화")\nprint(f"   - 4가지 사용자 세그먼트 커버")\nprint(f"   - 구체적인 질문 예시 포함")\n\nprint(f"\\n💡 참고:")\nprint(f"   - 기존 방식: 한국어 암묵적 기대 (실패 가능)")\nprint(f"   - 개선 방식: 한국어 명시적 지시 (성공률 ↑)")\nprint(f"   - RAGAS 0.3.1+에서 페르소나 기반 질문 생성 지원")


👥 사용자 페르소나 정의 (선택사항)...

✅ 4개 페르소나 정의 완료
   - 처음 이용자: 서울 대중교통을 처음 이용하는 사람. 기본적인 정보와 이용 방법이 필요함....
   - 출퇴근 통근자: 매일 지하철/버스로 출퇴근하는 직장인. 효율성과 정시성에 관심이 많음....
   - 어르신: 교통약자. 무료 승차, 편의시설, 접근성 정보가 필요함....
   - 관광객: 서울을 방문한 관광객. 주요 명소 접근 방법과 교통카드 구매 등의 정보 필요....

💡 참고: 페르소나는 TestsetGenerator 초기화 시 persona_list로 전달 가능
   현재는 사용하지 않지만, 필요시 Section 5.1에서 추가 가능


### 5.3 Knowledge Graph 생성 (RAGAS 0.3.1+)

RAGAS 0.3.1+의 Knowledge Graph 기반 접근 방식을 사용합니다.

#### Knowledge Graph의 역할
1. 문서를 구조화된 노드로 변환
2. Transform을 통해 메타데이터 추출 (headlines, keyphrases)
3. Query Synthesizer가 노드에서 질문 생성

In [ ]:
# ============================================================================
# Knowledge Graph 생성
# ============================================================================

from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from langchain_community.document_loaders import DirectoryLoader, TextLoader

print("\n🔨 Creating Knowledge Graph...")

# Load documents from crawled markdown
DOCS_DIR = ROOT_DIR / "data" / "crawled" / "seoul_traffic" / "markdown_deduplicated"

print(f"📄 Loading documents from: {DOCS_DIR}")
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
documents = loader.load()
print(f"✅ Loaded {len(documents)} documents")

# Create Knowledge Graph
kg = KnowledgeGraph()

for doc in documents:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": doc.page_content,
                "document_metadata": doc.metadata
            }
        )
    )

print(f"✅ Created Knowledge Graph with {len(kg.nodes)} nodes")

### 5.4 Transform Pipeline 적용

Knowledge Graph를 풍부하게 만들기 위한 Transform들을 적용합니다.

#### Transform 종류
1. **HeadlinesExtractor**: 섹션 제목 추출
2. **HeadlineSplitter**: 제목 기준으로 문서 분할
3. **KeyphrasesExtractor**: 핵심 키워드 추출

In [ ]:
# ============================================================================
# Transform Pipeline 적용
# ============================================================================

from ragas.testset.transforms import apply_transforms
from ragas.testset.transforms import HeadlinesExtractor, HeadlineSplitter, KeyphrasesExtractor

print("\n🔄 Applying transforms to Knowledge Graph...")

transforms = [
    HeadlinesExtractor(llm=generator_llm, max_num=20),
    HeadlineSplitter(max_tokens=1500),
    KeyphrasesExtractor(llm=generator_llm)
]

apply_transforms(kg, transforms=transforms)

print(f"✅ Transforms applied. Total nodes: {len(kg.nodes)}")
print(f"   - Headlines extracted and documents split")
print(f"   - Keyphrases extracted for query generation")

### 5.5 Query Synthesizer 설정

Knowledge Graph의 노드에서 질문을 생성하는 Query Synthesizer를 설정합니다.

#### Query Distribution Strategy
- **50% Headlines-based**: 구조화된 섹션 기반 질문
- **50% Keyphrases-based**: 핵심 키워드 기반 질문

In [ ]:
# ============================================================================
# Query Synthesizer 설정
# ============================================================================

from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

print("\n🎯 Configuring Query Synthesizers...")

query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(
            llm=generator_llm,
            property_name="headlines"
        ),
        0.5
    ),
    (
        SingleHopSpecificQuerySynthesizer(
            llm=generator_llm,
            property_name="keyphrases"
        ),
        0.5
    ),
]

print(f"✅ Query distribution configured:")
print(f"   - 50% Headlines-based queries")
print(f"   - 50% Keyphrases-based queries")

### 5.6 Korean Testset 생성

Knowledge Graph와 Personas를 사용하여 한국어 Q/A 쌍을 생성합니다.

#### Generation Process
1. Knowledge Graph에서 노드 선택
2. Persona 기반 질문 생성 (한국어 명시)
3. Query Synthesizer가 headlines/keyphrases에서 질문 생성
4. Ground truth 답변 추출

In [ ]:
# ============================================================================
# Korean Testset 생성
# ============================================================================

from ragas.testset import TestsetGenerator

print(f"\n🎯 Generating Korean testset...")
print(f"   - Target size: 50")
print(f"   - Personas: {len(personas)}")
print(f"   - Query distribution: Headlines (50%) + Keyphrases (50%)")

# Create TestsetGenerator with Knowledge Graph
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=embeddings,
    knowledge_graph=kg,
    persona_list=personas
)

# Generate testset
print(f"⏳ Generating 50 Korean Q/A pairs...")
testset = generator.generate(
    testset_size=50,
    query_distribution=query_distribution
)

# Convert to DataFrame
df_testset = testset.to_pandas()

print(f"\n✅ Korean testset generated!")
print(f"   - Total samples: {len(df_testset)}")
print(f"   - Columns: {df_testset.columns.tolist()}")

# Display sample
if len(df_testset) > 0:
    print(f"\n📝 Sample Korean Q/A:")
    sample = df_testset.iloc[0]
    print(f"   Question: {sample['user_input'][:100]}...")
    print(f"   Answer: {sample['reference'][:100]}...")

2025-10-30 17:35:22,330 - root - INFO - Starting RAGAS testset generation with cost tracking
2025-10-30 17:35:22,331 - root - INFO - Sample size: 200 documents
2025-10-30 17:35:22,332 - root - INFO - Target questions: 50 (using default_query_distribution)



🤔 질문 생성 시작...
   - 목표: 50개 질문 (default_query_distribution)
   - 예상 소요 시간: 5-10분
   - 예상 비용: ~$0.50-$1.00 (OpenAI API)


Applying HeadlinesExtractor:   0%|          | 0/181 [00:00<?, ?it/s]

2025-10-30 17:35:23,218 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: 'str' object has no attribute 'content'
2025-10-30 17:35:23,219 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: 'str' object has no attribute 'content'
2025-10-30 17:35:23,219 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: 'str' object has no attribute 'content'
2025-10-30 17:35:23,220 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: 'str' object has no attribute 'content'
2025-10-30 17:35:23,220 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: 'str' object has no attribute 'content'
2025-10-30 17:35:23,220 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: 'str' object has no attribute 'content'
2025-10-30 17:35:23,221 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: 'str' object has no attribute 'content'
2025-10-30 17:35:23,

Applying HeadlineSplitter:   0%|          | 0/200 [00:00<?, ?it/s]

...

2025-10-30 17:35:23,670 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
2025-10-30 17:35:23,671 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
2025-10-30 17:35:23,671 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
2025-10-30 17:35:23,671 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
2025-10-30 17:35:23,672 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
2025-10-30 17:35:23,672 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: node.property('summary') must be a string, found '

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

2025-10-30 17:35:23,822 - ragas.testset.transforms.engine - ERROR - unable to apply transformation: Node 3c9c8a75-1ef0-48ee-b4ad-0b3e671cd0a6 has no summary_embedding
2025-10-30 17:35:23,822 - root - ERROR - RAGAS testset generation failed: No nodes that satisfied the given filer. Try changing the filter.
Traceback (most recent call last):
  File "/var/folders/lg/zz8kzx7j04q6lvkcyzwjznwh0000gn/T/ipykernel_74687/4104452203.py", line 14, in <module>
    testset = testset_generator.generate_with_langchain_docs(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sdh/Dev/02_production_projects/humetro-ai-assistant/.venv/lib/python3.12/site-packages/ragas/testset/synthesizers/generate.py", line 188, in generate_with_langchain_docs
    return self.generate(
           ^^^^^^^^^^^^^^
  File "/Users/sdh/Dev/02_production_projects/humetro-ai-assistant/.venv/lib/python3.12/site-packages/ragas/testset/synthesizers/generate.py", line 369, in generate
    self.persona_list 


❌ 질문 생성 실패: No nodes that satisfied the given filer. Try changing the filter.

🔍 디버깅 정보:
   - sampled_docs 타입: <class 'list'>
   - sampled_docs 개수: 200
   - 첫 번째 doc 타입: <class 'langchain_core.documents.base.Document'>


ValueError: No nodes that satisfied the given filer. Try changing the filter.

### 5.7 한국어 품질 검증

생성된 Q/A 쌍이 한국어로 올바르게 생성되었는지 검증합니다.

In [ ]:
# ============================================================================
# 한국어 품질 검증
# ============================================================================

import re

print("\n🔍 한국어 품질 검증...")

def is_korean(text):
    """Check if text contains Korean characters"""
    if not text or not isinstance(text, str):
        return False
    korean_pattern = re.compile('[\u3131-\u3163\uac00-\ud7a3]+')
    return bool(korean_pattern.search(text))

korean_questions = df_testset['user_input'].apply(is_korean).sum()
korean_answers = df_testset['reference'].apply(is_korean).sum()
total = len(df_testset)

print(f"\n✅ 한국어 비율:")
print(f"   - 질문: {korean_questions}/{total} ({korean_questions/total*100:.1f}%)")
print(f"   - 답변: {korean_answers}/{total} ({korean_answers/total*100:.1f}%)")

if korean_questions < total * 0.9 or korean_answers < total * 0.9:
    print(f"\n⚠️  경고: 한국어 비율이 90% 미만입니다.")
    print(f"   - Persona 정의를 재확인하세요")
    print(f"   - '반드시 한국어로' 명시가 있는지 확인하세요")
else:
    print(f"\n🎉 한국어 품질 검증 통과!")

### 5.5 질문 품질 검증

생성된 질문의 품질을 검증합니다.

In [ ]:
print("\n🔍 질문 품질 검증...")

# 1. 한국어 질문 비율 확인
import re


def contains_korean(text):
    return bool(re.search("[\uac00-\ud7a3]", text))


korean_questions = df_testset["question"].apply(contains_korean).sum()
korean_ratio = korean_questions / len(df_testset) * 100

print(f"\n1️⃣ 한국어 질문 비율")
print(f"   - 한국어: {korean_questions}개 ({korean_ratio:.1f}%)")
print(f"   - 영어: {len(df_testset) - korean_questions}개 ({100 - korean_ratio:.1f}%)")

# 2. Ground truth 존재 여부
if "ground_truth" in df_testset.columns:
    has_ground_truth = df_testset["ground_truth"].notna().sum()
    gt_ratio = has_ground_truth / len(df_testset) * 100
    print(f"\n2️⃣ Ground Truth 존재")
    print(f"   - 있음: {has_ground_truth}개 ({gt_ratio:.1f}%)")
    print(f"   - 없음: {len(df_testset) - has_ground_truth}개")
else:
    print(f"\n2️⃣ Ground Truth 컬럼 없음")

# 3. 질문 길이 분포
question_lengths = df_testset["question"].str.len()
print(f"\n3️⃣ 질문 길이 분포")
print(f"   - 평균: {question_lengths.mean():.0f} chars")
print(f"   - 최소: {question_lengths.min()} chars")
print(f"   - 최대: {question_lengths.max()} chars")
print(f"   - 중앙값: {question_lengths.median():.0f} chars")

# 4. 질문 다양성 (고유 단어 수)
all_questions = " ".join(df_testset["question"].tolist())
unique_words = len(set(all_questions.split()))
total_words = len(all_questions.split())
diversity = unique_words / total_words * 100

print(f"\n4️⃣ 질문 다양성")
print(f"   - 총 단어: {total_words:,}개")
print(f"   - 고유 단어: {unique_words:,}개")
print(f"   - 다양성: {diversity:.1f}%")

print(f"\n✅ 질문 품질 검증 완료")

### 5.6 질문 저장

생성된 질문을 파일로 저장하여 재현성을 확보합니다.

In [ ]:
print("\n💾 질문 저장 중...")

# CSV 저장 (간단한 분석용)
csv_path = RESULTS_DIR / "ragas_questions_50.csv"
df_testset.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"   ✅ CSV 저장: {csv_path}")

# JSON 저장 (전체 데이터 + 재현성 + 비용)
json_path = RESULTS_DIR / "ragas_questions_50.json"
testset_dict = df_testset.to_dict(orient="records")

# 메타데이터 추가 (비용 정보 포함)
output_data = {
    "metadata": {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "total_questions": len(df_testset),
        "generator_llm": "gpt-5-mini",
        "critic_llm": "gpt-5-mini",
        "embeddings": "text-embedding-3-small",
        "sample_size": len(sampled_docs),
        "generation_time_seconds": generation_time,
        "korean_ratio": korean_ratio,
        "log_file": str(log_file),
        "total_cost_usd": total_cost,  # 비용 추가
        "cost_per_question_usd": total_cost / len(df_testset),  # 질문당 비용
        "query_distribution": "default_query_distribution",  # 분포 방식
    },
    "questions": testset_dict,
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"   ✅ JSON 저장: {json_path}")

# 로그 파일 복사 (결과와 함께 보관)
import shutil

log_copy_path = RESULTS_DIR / f"ragas_generation_{timestamp}.log"
shutil.copy2(log_file, log_copy_path)
print(f"   ✅ 로그 저장: {log_copy_path}")

print(f"\n📁 저장된 파일:")
print(f"   - CSV: {csv_path}")
print(f"   - JSON: {json_path}")
print(f"   - 로그: {log_copy_path}")

# 로그 통계
log_size_kb = log_copy_path.stat().st_size / 1024
print(f"\n📊 로그 파일 통계:")
print(f"   - 크기: {log_size_kb:.1f} KB")
print(f"   - 라인 수: {sum(1 for _ in open(log_copy_path, 'r', encoding='utf-8'))}")

# 비용 요약
print(f"\n💰 비용 요약:")
print(f"   - 총 비용: ${total_cost:.4f}")
print(f"   - 질문당 비용: ${total_cost / len(df_testset):.4f}")
print(f"   - 예상 절감: 수작업 대비 ~95%")

print("\n" + "=" * 80)
print("✅ Section 5 완료: RAGAS 질문 생성")
print("=" * 80)

print(f"\n📊 요약:")
print(f"   - 생성된 질문: {len(df_testset)}개")
print(f"   - 한국어 비율: {korean_ratio:.1f}%")
print(f"   - 소요 시간: {generation_time:.1f}초 ({generation_time / 60:.1f}분)")
print(f"   - 총 비용: ${total_cost:.4f}")
print(f"   - 결과 저장: CSV + JSON + 로그")

print(f"\n💡 로그 확인 방법:")
print(f"   - 명령어: tail -f {log_copy_path}")
print(f"   - 또는: cat {log_copy_path} | grep -i error")
print(f"   - 전체 보기: cat {log_copy_path}")

print(
    f"\n🎯 다음 단계: Section 6 - Naive Q/A System (4 models × 50 questions = 200 answers)"
)
print("=" * 80)

# 로깅 종료
logging.info("Section 5 completed successfully")
logging.info(f"Output files: CSV={csv_path}, JSON={json_path}, LOG={log_copy_path}")
logging.info(f"Total cost: ${total_cost:.4f} ({len(df_testset)} questions)")

### 5.7 로그 분석 도구

생성된 RAGAS 로그를 분석하기 위한 유틸리티입니다.

In [ ]:
# ============================================================================
# RAGAS 로그 분석 도구
# ============================================================================


def analyze_ragas_log(log_path):
    """
    RAGAS 로그 파일을 분석하여 주요 정보를 추출합니다.

    Args:
        log_path: 로그 파일 경로
    """
    import re
    from collections import Counter

    print(f"\n📊 로그 분석: {log_path}")
    print("=" * 80)

    if not Path(log_path).exists():
        print(f"❌ 로그 파일을 찾을 수 없습니다: {log_path}")
        return

    with open(log_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # 로그 레벨 통계
    log_levels = Counter()
    errors = []
    warnings = []
    api_calls = []

    for line in lines:
        # 로그 레벨 카운트
        if " - DEBUG - " in line:
            log_levels["DEBUG"] += 1
        elif " - INFO - " in line:
            log_levels["INFO"] += 1
        elif " - WARNING - " in line:
            log_levels["WARNING"] += 1
            warnings.append(line.strip())
        elif " - ERROR - " in line:
            log_levels["ERROR"] += 1
            errors.append(line.strip())

        # API 호출 추적
        if "openai" in line.lower() or "api" in line.lower():
            api_calls.append(line.strip())

    # 1. 로그 기본 정보
    print(f"\n1️⃣ 로그 기본 정보")
    print(f"   - 총 라인 수: {len(lines):,}")
    print(f"   - 파일 크기: {Path(log_path).stat().st_size / 1024:.1f} KB")

    # 2. 로그 레벨 분포
    print(f"\n2️⃣ 로그 레벨 분포")
    for level, count in log_levels.most_common():
        print(f"   - {level}: {count:,}개")

    # 3. 에러 메시지
    if errors:
        print(f"\n3️⃣ 에러 메시지 ({len(errors)}개)")
        for i, error in enumerate(errors[:5], 1):  # 최대 5개만 출력
            print(f"   {i}. {error[:100]}...")
        if len(errors) > 5:
            print(f"   ... ({len(errors) - 5}개 더 있음)")
    else:
        print(f"\n3️⃣ 에러 없음 ✅")

    # 4. 경고 메시지
    if warnings:
        print(f"\n4️⃣ 경고 메시지 ({len(warnings)}개)")
        for i, warning in enumerate(warnings[:5], 1):
            print(f"   {i}. {warning[:100]}...")
        if len(warnings) > 5:
            print(f"   ... ({len(warnings) - 5}개 더 있음)")
    else:
        print(f"\n4️⃣ 경고 없음 ✅")

    # 5. API 호출 통계
    print(f"\n5️⃣ API 호출 관련 로그")
    print(f"   - API 관련 로그: {len(api_calls)}개")
    if api_calls:
        print(f"   - 예시 (최근 3개):")
        for call in api_calls[-3:]:
            print(f"     • {call[:120]}...")

    print("\n" + "=" * 80)
    print("\n💡 상세 로그 확인:")
    print(f"   - 전체 보기: cat {log_path}")
    print(f"   - 에러만: grep -i error {log_path}")
    print(f"   - 경고만: grep -i warning {log_path}")
    print(f"   - API 호출: grep -i 'openai\|api' {log_path}")


# 사용 예시 (Section 5 실행 후 사용)
# analyze_ragas_log(RESULTS_DIR / "ragas_generation_YYYYMMDD_HHMMSS.log")

print("✅ 로그 분석 도구 준비 완료")
print("\n사용 방법:")
print('   analyze_ragas_log(RESULTS_DIR / "ragas_generation_YYYYMMDD_HHMMSS.log")')